# Results NVV-Pipeline Evaluation Experiment - Full-GT and Part-GT
- yml: environment.yml
- env: nvv_isolation_pipeline
- last updated: 05.04.2026


## Setup
- Imports
- Config

In [ ]:
#Setup
import os
import sys
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

results_dir = project_root / "data" / "results"

import pandas as pd
import matplotlib.pyplot as plt

from config.load_config import load_config
from config.path_factory import print_paths

from evaluation.analysis_loader import load_and_compare_workspaces

from evaluation.analysis_tables import build_rq1_capability_tables

from evaluation.analysis_tables import (
    build_setting_summary_table,
    build_best_all_screened_values_matrix,
    build_derivative_matrix,
    build_top_k_runs_table
)
from evaluation.analysis_plots import (
    plot_setting_overview,
    plot_derivative_group_comparison,
    plot_top_k_runs_per_setting,
)

cfg_path_nvs_full_gt = project_root / "experiments" / "pipeline_evaluation" / "config_nvs38k_full_gt.yaml"
cfg_path_nvs_part_gt = project_root / "experiments" / "pipeline_evaluation" / "config_nvs38k_part_gt.yaml"
cfg_path_vocal_part_gt = project_root / "experiments" / "pipeline_evaluation" / "config_vocal_part_gt.yaml"


config = load_config(cfg_path_nvs_full_gt)
print_paths(cfg_path_nvs_full_gt)

## Collect Pipeline Evaluation Experiment Results 
- Experiment specs for comparison


In [ ]:
# specification of full run evaluation experiments to load and compare (all settings)
specs = [
    {
        "label": "NVS-38K_EN | full_gt",
        "cfg_path": cfg_path_nvs_full_gt,
        "dataset_name": "NVS-38K_EN",
        "mode": "full_gt",
    },
    {
        "label": "NVS-38K_EN | part_gt",
        "cfg_path": cfg_path_nvs_part_gt,
        "dataset_name": "NVS-38K_EN",
        "mode": "part_gt",
    },
    {
        "label": "VOCAL_RA1 | part_gt",
        "cfg_path": cfg_path_vocal_part_gt,
        "dataset_name": "VOCAL_RA1",
        "mode": "part_gt",
    },
    {
        "label": "VOCAL_RA2 | part_gt",
        "cfg_path": cfg_path_vocal_part_gt,
        "dataset_name": "VOCAL_RA2",
        "mode": "part_gt",
    },
]

analysis_bundle = load_and_compare_workspaces(
    specs=specs,
    param_names=[],
    param_pairs=[],
)

bundles_by_spec = analysis_bundle["bundles_by_spec"]
views_by_spec = analysis_bundle["views_by_spec"]
views = analysis_bundle["combined_views"]
setting_order = analysis_bundle["setting_order"]


In [ ]:
bundles_by_spec.keys()

## RQ1 Pipeline Capability

In [ ]:

# RQ1 Pipeline Capability
# --- collect RQ1 tables from the loaded settings ---
rq1_all = pd.concat(
    [
        bundle["results"]["rq1"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq1_tables = build_rq1_capability_tables(rq1_all)

print("RQ1 Capability Table (Full-GT):")
display(rq1_tables["full_gt"])

print("RQ1 Capability Table (Part-GT):")
display(rq1_tables["part_gt"])




In [ ]:
# RQ1 appendix table for Word/Excel
full = rq1_tables["full_gt"].copy()
part = rq1_tables["part_gt"].copy()

systems_order = ["Baseline", "Best Selected Set", "Δ vs Baseline"]

# keep only relevant rows and order them
full = full[full["System"].isin(systems_order)].copy()
part = part[part["System"].isin(systems_order)].copy()

full["System"] = pd.Categorical(full["System"], categories=systems_order, ordered=True)
part["System"] = pd.Categorical(part["System"], categories=systems_order, ordered=True)

full = full.sort_values(["Setting", "System"])
part = part.sort_values(["Setting", "System"])

# select settings
nvs_full = full[full["Setting"] == "NVS-38K_EN | full_gt"]
nvs_part = part[part["Setting"] == "NVS-38K_EN | part_gt"]
ra1 = part[part["Setting"] == "VOCAL_RA1 | part_gt"]
ra2 = part[part["Setting"] == "VOCAL_RA2 | part_gt"]

# build final table with grouped headers
rq1_appendix = pd.DataFrame({
    ("", "System"): systems_order,

    ("NVS-38K_EN | full_gt", "F1"): nvs_full["F1"].tolist(),
    ("NVS-38K_EN | full_gt", "Recall"): nvs_full["Recall"].tolist(),
    ("NVS-38K_EN | full_gt", "EOS Recall"): nvs_full["EOS Recall"].tolist(),

    ("NVS-38K_EN | part_gt", "Recall"): nvs_part["Recall"].tolist(),
    ("NVS-38K_EN | part_gt", "EOS Recall"): nvs_part["EOS Recall"].tolist(),

    ("VOCAL_RA1 | part_gt", "Recall"): ra1["Recall"].tolist(),
    ("VOCAL_RA1 | part_gt", "EOS Recall"): ra1["EOS Recall"].tolist(),

    ("VOCAL_RA2 | part_gt", "Recall"): ra2["Recall"].tolist(),
    ("VOCAL_RA2 | part_gt", "EOS Recall"): ra2["EOS Recall"].tolist(),
})

rq1_appendix.columns = pd.MultiIndex.from_tuples(rq1_appendix.columns)

# round only for presentation
rq1_appendix = rq1_appendix.round(3)

rq1_appendix

In [ ]:
display(
    rq1_appendix
    .round(3)
    .style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "right"
    })
)

In [ ]:
rq1_path = results_dir / "rq1_capability_appendix_table.xlsx"


with pd.ExcelWriter(rq1_path, engine="openpyxl") as writer:
    rq1_appendix.to_excel(writer, sheet_name="RQ1 Capability", merge_cells=True)

print(rq1_path)

In [ ]:
from evaluation.rq_plots import plot_rq1_full_gt_grouped_bars, plot_rq1_part_gt_grouped_bars

fig = plot_rq1_full_gt_grouped_bars(rq1_tables["full_gt"])
plt.show()

fig = plot_rq1_part_gt_grouped_bars(
    rq1_tables["part_gt"],
    setting_order=setting_order,
)
plt.show()

## RQ2 Ranking Configuration: Single best

In [ ]:
from evaluation.analysis_tables import build_rq2a_single_ranking_tables

rq2a_single_all = pd.concat(
    [
        bundle["results"]["rq2a_single"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_single_tables = build_rq2a_single_ranking_tables(
    rq2a_single_all,
    top_k=10,
)

print("RQ2a Single Ranking Tables:")
for setting_name, df_table in rq2a_single_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
# examination of difference in mean dice eos recall and mean dice eos tp 
# --> identical in NVS dataset, because there is only one gt-event per clip 
# (either hit or miss, no partial credit possible)
rq2a_single_all[[
    "setting",
    "macro_mean_dice_eos_recall",
    "macro_mean_mean_dice_eos_tp",
]].head(100)

In [ ]:

from evaluation.rq_plots import plot_rq2a_rank_vs_score

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=42,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_recall",
    top_k=42,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=42,
    setting_order=setting_order,
)
plt.show()


## RQ2 Configuration Combination: Selected Set

In [ ]:
from evaluation.analysis_tables import build_rq2a_selected_set_tables

rq2a_selected_all = pd.concat(
    [
        bundle["results"]["rq2a_selected_set"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_selected_tables = build_rq2a_selected_set_tables(
    rq2a_single_all,
    rq2a_selected_all,
)

print("RQ2a Selected Set Tables:")
for setting_name, df_table in rq2a_selected_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

## RQ2 Audio Derivative Aggregation

In [ ]:
# RQ2 Audio Derivative Aggregation - Just for inspection, not for final presentation. ToDo: update artifact with insertion rate and remove precision and f1 for part_gt
# Keep derivative matrices separated by metric / mode
derivative_matrix_f1 = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1", #toDo: also recall and dice
)

derivative_matrix_recall = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall", #toDo also eos and eos_tp and insertion_rate
)

print("\nDerivative Matrix (F1, full_gt):")
display(derivative_matrix_f1)
print("\nDerivative Matrix (Recall, part_gt):")
display(derivative_matrix_recall)



In [ ]:
# RQ2b Audio Derivative Aggregation - Group Comparison (Thesis Tables)
from evaluation.analysis_tables import build_rq2b_derivative_tables
## RQ2b Audio Derivative Aggregation

rq2b_tables = build_rq2b_derivative_tables(views["derivative_comparison"])

print("RQ2b Audio Derivative Table (Full-GT):")
display(rq2b_tables["full_gt"])

print("RQ2b Audio Derivative Table (Part-GT):")
display(rq2b_tables["part_gt"])

In [ ]:
from evaluation.rq_plots import plot_rq2b_boxplot_with_points
fig = plot_rq2b_boxplot_with_points(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2b_boxplot_with_points(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

## RQ2b Inspection of top 2

In [ ]:
from evaluation.analysis_tables import inspect_top_n_rq2a_configs_by_group

df_top2_derivatives_full = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="full_gt",
    group_by="audio_derivative_group",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_derivatives_full)

df_top2_derivatives_part = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="part_gt",
    group_by="audio_derivative_group",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_derivatives_part)

## Configuration VAD Mask Inspection

In [ ]:
from evaluation.analysis_tables import build_rq2b_vad_mask_tables

rq2b_vad_tables = build_rq2b_vad_mask_tables(
    rq2a_single_all,
    setting_order=setting_order,
)

display(rq2b_vad_tables["full_gt"])
display(rq2b_vad_tables["part_gt"])

In [ ]:
from evaluation.rq_plots import plot_rq2b_vad_mask_boxplot_with_points

fig = plot_rq2b_vad_mask_boxplot_with_points(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2b_vad_mask_boxplot_with_points(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

In [ ]:
df_top2_vad_full = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="full_gt",
    group_by="vad_mask",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_vad_full)

df_top2_vad_part = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="part_gt",
    group_by="vad_mask",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_vad_part)

## RQ3 NVV Coverage

In [ ]:
## RQ3 NVV Coverage

from evaluation.analysis_tables import (
    build_rq3_full_gt_label_tables,
    build_rq3_part_gt_event_tables,
    build_rq3_global_tables,
)

rq3_label_all = pd.concat(
    [
        bundle["results"]["rq3_label"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq3_global_all = pd.concat(
    [
        bundle["results"]["rq3_global"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq3_full_gt_label_tables = build_rq3_full_gt_label_tables(rq3_label_all)
rq3_part_gt_event_tables = build_rq3_part_gt_event_tables(rq3_label_all)
rq3_global_tables = build_rq3_global_tables(rq3_global_all)



## RQ3 Global Metrics Tables - Full-GT and Part-GT

In [ ]:
display(rq3_global_tables["full_gt"])
display(rq3_global_tables["part_gt"])

## RQ3 Label Coverage (Full-GT) Table and Plot

In [ ]:
print("RQ3 Event Tables (Full-GT):")
for setting_name, df_table in rq3_full_gt_label_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
from evaluation.rq_plots import plot_rq3_label_coverage

fig = plot_rq3_label_coverage(
    rq3_full_gt_label_tables["NVS-38K_EN | full_gt"],
    setting=None,
)
plt.show()


## RQ3 Label quality (full-GT)

In [ ]:
from evaluation.rq_plots import plot_rq3_label_quality

fig = plot_rq3_label_quality(
    rq3_full_gt_label_tables["NVS-38K_EN | full_gt"]
)
plt.show()

## RQ3 Global Recall Comparison (Full-GT - Part-GT)

In [ ]:
from evaluation.rq_plots import plot_rq3_global_recall_comparison

fig = plot_rq3_global_recall_comparison(
    rq3_global_all,
    setting_order=setting_order,
)
plt.show()

## RQ3 Event Samples (Part-GT) Tables

In [ ]:
## RQ3 Event Samples (Part-GT) Tables
print("RQ3 Event Tables (Part-GT):")
for setting_name, df_table in rq3_part_gt_event_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
display(rq1_tables["full_gt"].columns)
display(rq2a_single_all.columns)
display(rq3_global_tables["full_gt"].columns)

| RQ | Experiment | Full-GT Mode | Part-GT Mode | Result |
|---|---|---|---|---|
| RQ1 Capability | Baseline vs. best configuration set | Greedy forward selection + union evaluation (F1-based) | Reuse of selected set + union evaluation | Best achievable score vs. baseline |
| RQ2 Sensitivity | Configuration ranking (single best) | Ranking based on full-GT metrics | Ranking based on part-GT metrics | Best single configuration |
|  | Configuration combination (selected set) | Greedy forward selection + union evaluation | Reuse of selected set + union evaluation | Best-performing configuration set |
|  | Audio derivative aggregation | Group-wise comparison by ASR input audio derivative | Distribution-based comparison across groups | Best-performing audio derivative group |
| RQ3 NVV Coverage | Coverage analysis | Global and label-level analysis | Global and Event-level analysis | Coverage across categories or events |

| RQ | Experiment | Full-GT Mode | Part-GT Mode | Result |
|---|---|---|---|---|
| RQ1 Capability | Compare baseline vs. best configuration set | Baseline evaluation + greedy forward selection (F1-based) with union evaluation | Baseline evaluation + reuse of selected configuration set with union evaluation | Best achievable score of the pipeline vs. baseline |
| RQ2 Sensitivity: Configuration Selection | Single: Rank all configurations; identify best single configuration | Configuration ranking based on full-GT metrics | Configuration ranking based on part-GT metrics | Best single configuration |
|  | Selected Set: Select complementary configurations via greedy forward selection and union | Greedy forward selection (F1-based) + union evaluation | Reuse of selected configuration set + union evaluation | Best-performing configuration set |
| RQ2 Sensitivity: Configuration Component (Audio Derivatives) | Aggregate configurations by ASR input audio derivative | Comparison of configuration performance distributions and best configurations per group | Comparison of configuration performance distributions and best configurations per group | Best-performing audio derivative group |
| RQ3 NVV Coverage | Analyze detected, missed, and inserted NVVs | Global metric-based and label-level analysis using full-GT metrics | Global and Event-level analysis of detected, missed, and inserted events | Coverage per category or event-level coverage representation |